# Loading pdf files and Vectorize the Conent

In [1]:
from pypdf import PdfReader
import os

In [2]:
def load_pdf_content(pdf_file):
    try:
        reader = PdfReader(pdf_file)

        total_pages = len(reader.pages)
        # print('Total Pages:',total_pages)

        # print('Page 1:',page_1.extract_text())
        pages = []
        for p in range(total_pages):
            page = reader.get_page(p)
            content = page.extract_text()
            content = post_extract_processing(content)
            pages.append(content)
    except:
        print('Failed to load ', pdf_file)
        return None

    return pages


def post_extract_processing(text_content):
    lines = text_content.split('\n')
    # print(len(lines))
    new_lines = []
    for i in range(len(lines)):
        # print(len(lines[i]))
        if new_lines:
            line_1 = new_lines[-1]
        else:
            line_1 = ''
        line_2 = lines[i]
        last_of_line_1 = line_1[-1] if line_1 else ''
        if last_of_line_1 in ['-',',',' ',';','"'] or last_of_line_1.isalpha():
            line_1 = line_1[:-1] if line_1[-1] == '-' else line_1
            new_lines[-1] = line_1 +(' ' if last_of_line_1.isalpha else '') +  line_2
        else:
            new_lines.append(line_2)
    return new_lines #Paragraph

In [9]:
import requests
import json
from pymed import PubMed

url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pmc&id={pmc_id}&retmode=json'

pubmed = PubMed(tool="MyTool", email="qzhao9@lakeheadu.ca")

def get_pmc_metadata(pmc_ids):
    """
    Query document meta data
    """
    ids = ','.join([id.lstrip('PMC') for id in pmc_ids])
    try:
        results = pubmed._get("/entrez/eutils/esummary.fcgi",parameters={'db':'pmc','id':ids,'retmode':'json'})
        meta_result = results['result']
        uuids = meta_result['uids']
        metadata = {}
        for uuid in uuids:
            result = meta_result[uuid]
            authors = ', '.join([d['name'] for d in result['authors']])
            dois = [id['value'] for id in result['articleids'] if id['idtype']=='doi']
            if any(dois):
                doi = dois[0]
            # print('published On:', result['pubdate'])
            # print('authors', authors)
            # print('Title:', result['title'])
            # print(f"journal: {result['fulljournalname']} volume {result['volume']} issue {result['issue']} pages {result['pages']}")
            # print(f"article id: https://doi.org/{doi}")
            dois = '; '.join([f"{id['idtype'].upper()}:{id['value']}" for id in result['articleids']])
            cite_nlm = f"{authors}. {result['title']}. {result['source']}. {result['printpubdate']}; {result['volume']}({result['issue']}):{result['pages']}. Epub {result['epubdate']}. {dois}."
            # print(cite_nlm)
            metadata[f'PMC{uuid}'] = {
                             'id':f'PMC{uuid}',
                             'authors':authors,
                             'title': result['title'],
                             'doi':doi,
                             'cite':cite_nlm}
    except Exception as e:
        print('Failed to get meta data, error: ', e)
        return None

    return metadata


In [10]:
import os
import time
from tqdm import tqdm

data_dir = r'C:\COMP9800\Dataset\PMC-Figures\files'
metadata_file = data_dir + r'\metadata.jsonl'

file_folders = os.listdir(data_dir)
extracted_docs = []
steps = 200
file_folders = [folder for folder in file_folders if os.path.isdir(os.path.join(data_dir,folder))]
for i in tqdm(range(0, len(file_folders), steps)):
    batch = file_folders[i:i+steps]
    metadata = get_pmc_metadata(batch)
    for folder in tqdm(batch):
        full_path = os.path.join(data_dir,folder)
        files = os.listdir(full_path)
        num_of_pdf = len([f for f in files if f.endswith('.pdf') ])
        if num_of_pdf == 0:
            # print(folder,'No pdf file.')
            continue

        nxml_file = [f for f in files if f.endswith('.nxml')]
        if any(nxml_file):
            pdf_file = nxml_file[0].replace('.nxml','.pdf')
            ful_pdf = os.path.join(full_path,pdf_file)
            if os.path.isfile(ful_pdf):
                content = load_pdf_content(ful_pdf)
                if not content:
                    continue
                meta = metadata[folder]
                meta['content'] = content
                extracted_docs.append(meta)


  0%|          | 0/10 [00:00<?, ?it/s]EOF marker not found


Failed to load  C:\COMP9800\Dataset\PMC-Figures\files\PMC10343926\molecules-28-05081.pdf


 30%|███       | 3/10 [18:54<45:02, 386.04s/it]parsing for Object Streams
parsing for Object Streams
parsing for Object Streams
 40%|████      | 4/10 [26:15<40:44, 407.47s/it]PdfReadError("Invalid Elementary Object starting with b'\\xa7' @3771880: b'\\xb5;\\xab\\\\rH\\xd6\\xb7V\\x93\\x17\\xe3\\xf9^\\xe8) (])\\xa7-\\x98\\x13[!L[e%D]\\xc1\\xe1\\\\() ]\\r\\n/Root 85 0 R\\r\\n/Info 125 0 R\\r\\n/Size 195\\r'")
Cannot find "/Root" key in trailer
Searching object with "/Catalog" key


Failed to load  C:\COMP9800\Dataset\PMC-Figures\files\PMC4059717\pone.0100036.pdf


 50%|█████     | 5/10 [31:01<30:19, 363.91s/it]Multiple definitions in dictionary at byte 0x18fca for key /MediaBox
Multiple definitions in dictionary at byte 0x1922d for key /MediaBox
Multiple definitions in dictionary at byte 0x1940b for key /MediaBox
Multiple definitions in dictionary at byte 0x195b1 for key /MediaBox
Multiple definitions in dictionary at byte 0x1978f for key /MediaBox
Multiple definitions in dictionary at byte 0x199a5 for key /MediaBox
Multiple definitions in dictionary at byte 0x19bd6 for key /MediaBox
Multiple definitions in dictionary at byte 0x19dff for key /MediaBox
Multiple definitions in dictionary at byte 0x3807a for key /MediaBox
Multiple definitions in dictionary at byte 0x382b4 for key /MediaBox
Multiple definitions in dictionary at byte 0x3854e for key /MediaBox
Multiple definitions in dictionary at byte 0x3874d for key /MediaBox
Multiple definitions in dictionary at byte 0x38944 for key /MediaBox
Multiple definitions in dictionary at byte 0x38bd6 for k

...

 80%|████████  | 8/10 [48:44<12:06, 363.30s/it]Multiple definitions in dictionary at byte 0x1e287 for key /MediaBox
Multiple definitions in dictionary at byte 0x1e4c8 for key /MediaBox
Multiple definitions in dictionary at byte 0x1e630 for key /MediaBox
Multiple definitions in dictionary at byte 0x1e78f for key /MediaBox
Multiple definitions in dictionary at byte 0x1e906 for key /MediaBox
Multiple definitions in dictionary at byte 0x1eac6 for key /MediaBox
Multiple definitions in dictionary at byte 0x1ec86 for key /MediaBox
Multiple definitions in dictionary at byte 0x1ee6e for key /MediaBox
Multiple definitions in dictionary at byte 0x1f05d for key /MediaBox
Multiple definitions in dictionary at byte 0x1f2dc for key /MediaBox
Multiple definitions in dictionary at byte 0x1f4b3 for key /MediaBox
 90%|█████████ | 9/10 [56:07<06:28, 388.36s/it]EOF marker not found


Failed to load  C:\COMP9800\Dataset\PMC-Figures\files\PMC9598597\antioxidants-11-01947.pdf


incorrect startxref pointer(4)
parsing for Object Streams
Object 76 0 not defined.


Failed to load  C:\COMP9800\Dataset\PMC-Figures\files\PMC9875997\ct9-14-e00545.pdf


100%|██████████| 10/10 [59:38<00:00, 357.84s/it]


In [13]:
import json

with open(metadata_file,'w+') as file:
    json.dump(extracted_docs,file)

In [19]:
with open(metadata_file,'r+') as docfile:
    content = json.load(docfile)

content[0]['content'][0]

['Citation: Wang, K.; Qin, L.; Cao, J.; Zhang, L.; Liu, M.; Qu, C.; Miao, J.',
 'κ-Selenocarrageenan Oligosaccharides Prepared by Deep-Sea Enzyme Alleviate Inﬂammatory Responses and Modulate Gut Microbiota in Ulcerative Colitis Mice. Int. J. Mol.',
 'Sci. 2023, 24, 4672. https://doi.org/',
 '10.3390/ijms24054672',
 'Academic Editor: Daniel Plano Amatriain Received: 4 February 2023',
 'Revised: 13 February 2023',
 'Accepted: 16 February 2023',
 'Published: 28 February 2023',
 'Copyright: © 2023 by the authors.',
 'Licensee MDPI, Basel, Switzerland.',
 'This article is an open access article distributed under the terms and conditions of the Creative Commons Attribution (CC BY) license (https://',
 'creativecommons.org/licenses/by/',
 '4.0/).',
 ' International Journal of  Molecular Sciences Article κ-Selenocarrageenan Oligosaccharides Prepared by Deep-Sea Enzyme Alleviate Inﬂammatory Responses and Modulate Gut Microbiota in Ulcerative Colitis Mice Kai Wang 1,2 , Ling Qin 2, Junhan Cao 1,

In [26]:
corpus = []
docs = [c['content'] for c in content]
for doc in docs:
    for lines in doc:
        corpus.extend(lines)
    # print(len(doc))
    # print(doc)
    # break

# Build FAISS index

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. Load embedding model
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# 2. Compute embeddings
embeddings = embedder.encode(corpus, convert_to_numpy=True)

# 3. Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)


c:\Users\50183\.conda\envs\pdf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# meta = reader.metadata

# # All the following could be None!
# print('Title:',meta.title)
# print('Author:',meta.author)
# print('Subject:',meta.subject)
# print('keywords:',meta.keywords)
# # print(meta.producer)
# print('Created On:',meta.creation_date)
# print(meta.modification_date)

Title: Incidence and Long‐Term Outcomes of Pouchitis in Ulcerative Colitis Patients
Author: None
Subject: None
keywords: None
Created On: 2025-11-04 12:22:10+05:30
2025-11-04 23:50:04+05:30


# pdfplumber

In [ ]:
# import pdfplumber
# with pdfplumber.open(pdf_file) as pdf:
#     first_page = pdf.pages[5]
#     text = first_page.extract_text()
#     table = first_page.extract_table()
#     image = first_page.images
#     print(text) 
#     print(table)
#     print(image)

None
[]


# "unstructured[all-docs]"

In [ ]:
# from unstructured.partition.auto import partition
# blocks = partition(filename=pdf_file)
# for block in blocks:
#     print(f"{block.category}: {block.text}")

c:\Users\50183\.conda\envs\pdf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UncategorizedText: Article The Relationship Between Disease Activity and Fecal Calprotectin and Fecal Occult Blood in Inflammatory Bowel Disease: The Role of Nutritional Status
Title: Ali Bilgen 1,* and Hale Akpınar 2
UncategorizedText: 1 Department of Gastroenterology, Private Anka Hospital, ¸Sehitkamil District, 99 Number Street, No. 162,
Title: Gaziantep 27590, Turkey
NarrativeText: 2 Department of Gastroenterology, Faculty of Medicine, Dokuz Eylul University, ˙Izmir 35340, Turkey * Correspondence: dralibilgen@hotmail.com; Tel.: +90-342-329-27-27 or +90-532-507-0802
Title: Abstract
Title: Academic Editor: Kai Wang
Title: Received: 4 October 2025
Title: Revised: 23 October 2025
Title: Accepted: 24 October 2025
Title: Published: 28 October 2025
NarrativeText: Background: Inflammatory bowel disease (IBD), encompassing ulcerative colitis (UC) and Crohn’s disease (CD), is characterized by chronic intestinal inflammation with fluctuating clinical severity. Although fecal calprotectin (FC)